# CliffForge on Dablander's Mpro matched molecular pairs: final five-fold run

In Google Colab, select **Runtime → Change runtime type → GPU**, set **Runtime Version → 2026.07** (Python 3.12), then run **Runtime → Run all**. ChemProp 1.6.1 runs in a separate Python environment with NumPy 1.26; that version has no wheel for Colab’s newer Python 3.13 runtime. This is one self-contained notebook; it does not need the older Mpro notebooks. If an earlier run failed while creating the ChemProp environment, rerunning this updated notebook repairs that partial environment. It downloads a commit-pinned copy of Dablander's cleaned 1,924-molecule and 12,594-MMP source files, checks their schema, endpoint mapping and derived label counts, runs the same five molecule folds for CliffForge, ChemProp and 5-NN, and downloads one ZIP containing raw input hashes, splits, training logs, every model checkpoint, per-molecule and per-pair predictions, metrics, bootstrap draws and a readable report. A partial ZIP is offered if the experiment raises an error.

**Prespecified protocol**

- Original Dablander MMPs only. A cliff is an MMP with measured `|ΔpIC50| ≥ 2` (≥100-fold); a non-cliff has `|ΔpIC50| ≤ 1` (≤10-fold). Intermediate MMPs are excluded **only from binary detection**. No Tanimoto cutoff, stereo removal, MMP deduplication, old 1.0 cliff label or old 35% series split.
- Five shuffled molecule folds, fixed seed 17017. In each fold, all three methods receive the same four outer-training folds and are evaluated on the same held-out MMP rows. CliffForge trains on Dablander MMPs whose endpoints are both in those four folds; 5-NN uses all four folds. ChemProp receives those same outer-training molecules and performs its version 1.6.1 default internal 80/10/10 train/validation/test split. No outer-test activity is supplied to training or prediction.
- Three seeds for CliffForge and three for ChemProp 1.6.1 per fold. CliffForge uses batch size 32, trains for all 70 epochs, and saves the final checkpoint, matching the earlier Mpro training schedule. ChemProp uses its version 1.6.1 default batch size 50 and chooses its best checkpoint by internal validation RMSE over 60 epochs. No settings are tuned on outer-test outcomes. The published 1.5-pIC50 decision threshold is fixed in advance. Binary cliff accuracy and confusion counts are reported alongside AP, ROC AUC, MCC, F1, precision and recall.
- **Strict detection:** both MMP molecules held out in the same fold. Primary metric: pooled out-of-fold average precision (AP) using the published binary labels.
- **Within-series ranking:** for each known outer-training reference and its held-out MMP neighbors, Spearman correlation between measured and predicted candidate activities; a group needs at least three distinct candidates with nonconstant measured activity. Primary metric: mean per-reference Spearman. This is a Dablander-native local analogue series; no structural-similarity threshold is added.
- **Overall relative affinity:** numerical ΔpIC50 prediction error and correlation across all held-out MMP evaluation rows, reported pooled, separately for test–test and test–train pairs, and for qualifying local-series reference neighborhoods. Direction accuracy is also reported.
- **5-NN:** Morgan radius-2, 2048-bit fingerprints without chirality and similarity-weighted five neighbors, matching the earlier Mpro baseline.
- **Lead optimization:** one held-out candidate and one measured outer-training reference. Select the top 10% of candidate/reference proposals per fold by predicted signed activity gain; primary metric: mean measured gain of selected proposals. For ChemProp and 5-NN the predicted gain is **predicted candidate pIC50 minus measured reference pIC50**. CliffForge predicts the pair difference directly.
- Preplanned statistical contrasts: CliffForge minus ChemProp and CliffForge minus 5-NN for each primary metric (six in total). Paired cluster bootstrap with 5,000 resamples and a Bonferroni familywise 0.05 interval. All six lower bounds must exceed zero for a joint superiority claim. This is an approximate conditional analysis of this Mpro dataset; a positive result is not guaranteed.

The model recipes follow the older Mpro run. The dataset-specific CliffForge training-pair input is Dablander’s supplied MMP table, as requested; the earlier Mpro run generated ordered within-series pairs instead. The Dablander cliff labels and evaluation rows remain source-native. Thus the two datasets’ scores are not a controlled cross-dataset estimate of dataset difficulty.

This is a **new five-fold experiment**, not an exact reproduction of Dablander et al.'s three repeats of two-fold CV. Source: [paper](https://link.springer.com/article/10.1186/s13321-023-00708-w) and [data repository](https://github.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments).


In [ ]:
import sys, subprocess, os, json, math, time, random, hashlib, shutil, traceback, platform
from pathlib import Path
from datetime import datetime, timezone

# ChemProp 1.6.1 uses NumPy 1.x APIs; Colab's Python 3.13 runtime cannot install
# NumPy 1.26. Select Colab Runtime Version 2026.07 (Python 3.12).
if sys.version_info >= (3, 13):
    raise RuntimeError('Select Colab Runtime Version 2026.07 (Python 3.12), then rerun the notebook.')
# Colab's preinstalled Torch supplies the matching CUDA build. Keep the
# notebook's NumPy separate from ChemProp 1.6.1's NumPy 1.26 requirement.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch-geometric==2.8.1', 'rdkit', 'pandas', 'scipy', 'scikit-learn'], check=True)
CP_VENV = Path('/content/cliffforge_chemprop_venv')
CP_PYTHON = CP_VENV / 'bin' / 'python'
CP_READY = CP_VENV / '.cliffforge_virtualenv_ready'
# Some Colab Python images omit ensurepip, so stdlib venv fails after creating
# a partial directory. virtualenv seeds pip independently of ensurepip.
if not (CP_PYTHON.is_file() and CP_READY.is_file()
        and CP_READY.read_text() == sys.version):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv==20.35.3'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--clear',
                    '--system-site-packages', str(CP_VENV)], check=True)
    assert CP_PYTHON.is_file(), 'ChemProp Python environment was not created'
    subprocess.run([str(CP_PYTHON), '-m', 'pip', '--version'],
                   check=True, stdout=subprocess.DEVNULL)
    CP_READY.write_text(sys.version)
subprocess.run([str(CP_PYTHON), '-m', 'pip', 'install', '-q',
                'numpy==1.26.4', 'pandas==2.2.3', 'chemprop==1.6.1',
                'scipy==1.15.3', 'scikit-learn==1.6.1'], check=True)
cp_versions = json.loads(subprocess.check_output([str(CP_PYTHON), '-c',
    'import json,numpy,pandas,importlib.metadata as md; print(json.dumps({\"chemprop\":md.version(\"chemprop\"),\"numpy\":numpy.__version__,\"pandas\":pandas.__version__}))'], text=True))
assert cp_versions == {'chemprop': '1.6.1', 'numpy': '1.26.4', 'pandas': '2.2.3'}, cp_versions

import numpy as np
import pandas as pd
import scipy
import sklearn
import rdkit
import torch
import torch_geometric
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from sklearn.model_selection import KFold
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             matthews_corrcoef, f1_score, precision_score, recall_score,
                             accuracy_score, mean_absolute_error, mean_squared_error)
from scipy.stats import spearmanr
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
import torch.nn as nn

ROOT = Path('/content/cliffforge_dablander_5fold')
ROOT.mkdir(parents=True, exist_ok=True)
FOLDS, SEEDS, SPLIT_SEED = 5, (0, 1, 2), 17017
REL_EPOCHS, CP_EPOCHS = 70, 60
REL_BATCH_SIZE, BOOTSTRAPS = 32, 5000
CLIFF_CUTOFF, NONCLIFF_CUTOFF, PRED_CUTOFF = 2.0, 1.0, 1.5
LEAD_BUDGET = 0.10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| Torch:', torch.__version__, '| PyG:', torch_geometric.__version__)
if DEVICE.type != 'cuda':
    print('WARNING: GPU is not available; the full run will be much slower.')

def dump_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

protocol = {
    'dataset': 'Dablander COVID Moonshot SARS-CoV-2 Mpro cleaned molecule and MMP tables',
    'source': 'https://github.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments',
    'source_commit': '3081a1a40e3f5105a60b6d8b41f1ce69f8a0f453',
    'source_counts_expected': {'molecules': 1924, 'mmps': 12594},
    'activity': 'pIC50 = 6 - log10(IC50 in micromolar)',
    'cliff': {'positive_abs_delta_gte': CLIFF_CUTOFF, 'negative_abs_delta_lte': NONCLIFF_CUTOFF,
              'intermediate': 'exclude only from binary detection', 'predicted_abs_delta_gt': PRED_CUTOFF},
    'splits': {'outer': 'molecule-level shuffled KFold', 'n_folds': FOLDS, 'seed': SPLIT_SEED,
               'chemprop_internal': 'ChemProp 1.6.1 default random 80/10/10 of outer train'},
    'model_seeds': list(SEEDS), 'relative_max_epochs': REL_EPOCHS,
    'relative_batch_size': REL_BATCH_SIZE, 'chemprop_version': '1.6.1',
    'chemprop_batch_size': 50, 'knn_fingerprint': 'Morgan radius 2, 2048 bits, no chirality',
    'embedded_model_code_sha256': {
        'src/cliffforge/features.py': '5951069a83e9909b23ca6b90517ec74079b31c9cbd8bd440f0c3bf01b7251150',
        'src/cliffforge/networks.py': 'e52ee2bb5efa557f36c77a2ae9f75ec0ab2795c3d4a49ffe30ad7fa232aa9b71'},
    'chemprop_max_epochs': CP_EPOCHS,
    'relative_checkpoint_selection': 'final checkpoint after 70 epochs',
    'relative_training_pairs': 'Dablander supplied MMP rows with both endpoints in outer train',
    'chemprop_checkpoint_selection': 'best internal validation RMSE over 60 epochs',
    'lead_top_fraction_per_fold': LEAD_BUDGET, 'bootstrap_resamples': BOOTSTRAPS,
    'primary_metrics': ['strict_test_test_AP', 'local_reference_mean_Spearman', 'lead_top10_mean_measured_delta'],
    'contrasts': 'CliffForge minus ChemProp and CliffForge minus 5-NN on each primary metric',
    'multiplicity': 'six two-sided Bonferroni-adjusted percentile cluster-bootstrap intervals, FWER 0.05',
}
dump_json(ROOT / 'protocol.json', protocol)
env = {'created_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version,
       'platform': platform.platform(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
       'torch_geometric': torch_geometric.__version__, 'rdkit': rdkit.__version__,
       'chemprop_environment': cp_versions,
       'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
       'sklearn': sklearn.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
dump_json(ROOT / 'environment.json', env)
with open(ROOT / 'pip_freeze.txt', 'w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)
with open(ROOT / 'chemprop_pip_freeze.txt', 'w') as f:
    subprocess.run([str(CP_PYTHON), '-m', 'pip', 'freeze'], stdout=f, check=True)


## CliffForge graph featurizer and Siamese model

The next two cells embed the project model definitions so this notebook is self-contained. Their SHA-256 hashes are recorded in the results bundle.


In [ ]:
"""Whole-molecule 2D graph featurization (ChemProp-style) used by the relative-affinity model.

Extracted unchanged from the internship research code (featurize.py); the fragment/ghost-anchor channels of that
research are not used by CliffForge, so the corresponding node dimensions stay zero. Keep NODE_DIM/EDGE_DIM
unchanged: they define the shapes stored in the model checkpoints.
"""
from __future__ import annotations

import numpy as np
from rdkit import Chem
from rdkit.Chem import rdPartialCharges

ATOM_LIST = [5, 6, 7, 8, 9, 15, 16, 17, 35, 53]  # B C N O F P S Cl Br I
DEGREE_LIST = [0, 1, 2, 3, 4, 5]
CHARGE_LIST = [-2, -1, 0, 1, 2]
NUMH_LIST = [0, 1, 2, 3, 4]
HYB_LIST = [
    Chem.HybridizationType.SP,
    Chem.HybridizationType.SP2,
    Chem.HybridizationType.SP3,
    Chem.HybridizationType.SP3D,
    Chem.HybridizationType.SP3D2,
]

N_GHOST_DESC = 5  # inductive, resonance, electronegativity, polarizability, excluded-volume


def _onehot(value, choices):
    vec = [0.0] * (len(choices) + 1)
    try:
        vec[choices.index(value)] = 1.0
    except ValueError:
        vec[-1] = 1.0
    return vec


def atom_base_features(atom: Chem.Atom) -> list:
    """Standard atom features shared by every node (real atoms AND ghosts)."""
    feats = []
    feats += _onehot(atom.GetAtomicNum(), ATOM_LIST)          # 11
    feats += _onehot(atom.GetTotalDegree(), DEGREE_LIST)      # 7
    feats += _onehot(atom.GetFormalCharge(), CHARGE_LIST)     # 6
    feats += _onehot(atom.GetTotalNumHs(), NUMH_LIST)         # 6
    feats += _onehot(atom.GetHybridization(), HYB_LIST)       # 6
    feats.append(1.0 if atom.GetIsAromatic() else 0.0)        # 1
    feats.append(1.0 if atom.IsInRing() else 0.0)             # 1
    gc = atom.GetDoubleProp("_GasteigerCharge") if atom.HasProp("_GasteigerCharge") else 0.0
    if not np.isfinite(gc):
        gc = 0.0
    feats.append(float(gc))                                   # 1
    return feats   # length 39


ATOM_BASE_DIM = 39
# node layout: base(39) + is_ghost(1) + ghost_desc(5) + local_xyz(3)
NODE_DIM = ATOM_BASE_DIM + 1 + N_GHOST_DESC + 3


def bond_features(bond: Chem.Bond, length: float = 0.0) -> list:
    """Edge features. The 5th flag marks a *virtual* ghost-anchor edge."""
    if bond is None:  # virtual ghost edge
        return [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, float(length)]
    btype = bond.GetBondType()
    return [
        1.0 if btype == Chem.BondType.SINGLE else 0.0,
        1.0 if btype == Chem.BondType.DOUBLE else 0.0,
        1.0 if btype == Chem.BondType.TRIPLE else 0.0,
        1.0 if btype == Chem.BondType.AROMATIC else 0.0,
        0.0,  # virtual flag
        1.0 if bond.GetIsConjugated() else 0.0,
        1.0 if bond.IsInRing() else 0.0,
        float(length),
    ]   # length 8


EDGE_DIM = 8


def compute_gasteiger(mol: Chem.Mol) -> None:
    try:
        rdPartialCharges.ComputeGasteigerCharges(mol)
    except Exception:
        for a in mol.GetAtoms():
            a.SetDoubleProp("_GasteigerCharge", 0.0)


def mol_to_whole_graph(mol: Chem.Mol) -> dict | None:
    """Featurize a whole molecule into a 2D topological graph (ChemProp-style).
    Ghost / 3D channels stay zero for the baseline."""
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    mol = Chem.Mol(mol)
    compute_gasteiger(mol)
    n = mol.GetNumAtoms()
    x = np.zeros((n, NODE_DIM), dtype=np.float32)
    for atom in mol.GetAtoms():
        x[atom.GetIdx(), :ATOM_BASE_DIM] = atom_base_features(atom)
    src, dst, eattr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        src += [i, j]
        dst += [j, i]
        eattr += [bf, bf]
    if len(src) == 0:
        src, dst, eattr = [0], [0], [bond_features(None)]
    return {
        "x": x,
        "edge_index": np.array([src, dst], dtype=np.int64),
        "edge_attr": np.array(eattr, dtype=np.float32),
    }


In [ ]:
"""D-MPNN encoder and Siamese relative-affinity network (extracted unchanged from the internship code, models.py)."""
from __future__ import annotations

import torch
import torch.nn as nn
from torch_geometric.utils import scatter
from torch_geometric.nn import global_mean_pool


class DMPNNEncoder(nn.Module):
    """Directed MPNN. Returns a graph-level embedding [B, hidden]."""

    def __init__(self, node_dim: int, edge_dim: int, hidden: int = 128,
                 depth: int = 4, dropout: float = 0.1):
        super().__init__()
        self.depth = depth
        self.W_i = nn.Linear(node_dim + edge_dim, hidden)
        self.W_h = nn.Linear(hidden, hidden)
        self.W_o = nn.Linear(node_dim + hidden, hidden)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.ReLU()

    @staticmethod
    def _reverse_index(E: int, device) -> torch.Tensor:
        # edges are built as consecutive (i->j, j->i) pairs => reverse = idx XOR 1
        idx = torch.arange(E, device=device)
        return idx ^ 1

    def forward(self, x, edge_index, edge_attr, batch, return_nodes=False):
        src, dst = edge_index[0], edge_index[1]
        E = edge_attr.size(0)
        N = x.size(0)
        rev = self._reverse_index(E, x.device)

        # initial directed-edge hidden state
        h0 = self.act(self.W_i(torch.cat([x[src], edge_attr], dim=1)))  # [E, H]
        h = h0
        for _ in range(self.depth - 1):
            agg = scatter(h, dst, dim=0, dim_size=N, reduce="sum")       # [N, H]
            m = agg[src] - h[rev]                                        # exclude reverse
            h = self.act(h0 + self.W_h(m))
            h = self.dropout(h)

        agg = scatter(h, dst, dim=0, dim_size=N, reduce="sum")           # [N, H]
        h_node = self.act(self.W_o(torch.cat([x, agg], dim=1)))          # [N, H]
        h_node = self.dropout(h_node)
        if return_nodes:
            return h_node                                               # [N, H]
        return global_mean_pool(h_node, batch)


class SiameseAVES(nn.Module):
    """Siamese fragment network predicting relative ddG between two ligands."""

    def __init__(self, node_dim, edge_dim, hidden=128, depth=4, dropout=0.1):
        super().__init__()
        self.encoder = DMPNNEncoder(node_dim, edge_dim, hidden, depth, dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def encode(self, data):
        return self.encoder(data.x, data.edge_index, data.edge_attr, data.batch)

    def forward(self, data_cand, data_ref):
        g_c = self.encode(data_cand)
        g_r = self.encode(data_ref)
        combined = torch.cat([g_c, g_r, torch.abs(g_c - g_r)], dim=1)
        return self.head(combined).squeeze(-1)


In [ ]:
from urllib.request import urlretrieve
BASE = 'https://raw.githubusercontent.com/MarkusFerdinandDablander/QSAR-activity-cliff-experiments/3081a1a40e3f5105a60b6d8b41f1ce69f8a0f453/data/postera_sars_cov_2_mpro'
raw_dir = ROOT / 'raw'; raw_dir.mkdir(exist_ok=True)
for name in ('molecule_data_clean.csv', 'MMP_data_clean.csv'):
    dest = raw_dir / name
    if not dest.exists():
        print('Downloading', name)
        partial = dest.with_suffix(dest.suffix + '.part')
        try:
            urlretrieve(f'{BASE}/{name}', partial)
            assert partial.stat().st_size > 1000, f'Download incomplete: {partial}'
            partial.replace(dest)
        finally:
            partial.unlink(missing_ok=True)
    assert dest.stat().st_size > 1000, f'Download incomplete: {dest}'

mol_raw = pd.read_csv(raw_dir / 'molecule_data_clean.csv')
mmp_raw = pd.read_csv(raw_dir / 'MMP_data_clean.csv')
assert len(mol_raw) == 1924, f'Unexpected source molecule count: {len(mol_raw)}'
assert len(mmp_raw) == 12594, f'Unexpected source MMP count: {len(mmp_raw)}'
assert {'SMILES', 'f_avg_IC50 [uM]'}.issubset(mol_raw.columns), mol_raw.columns.tolist()
assert {'smiles_1', 'smiles_2'}.issubset(mmp_raw.columns), mmp_raw.columns.tolist()
smiles = mol_raw['SMILES'].astype(str).tolist()
ic50 = pd.to_numeric(mol_raw['f_avg_IC50 [uM]'], errors='raise').to_numpy(float)
assert np.isfinite(ic50).all() and (ic50 > 0).all(), 'Source has invalid IC50 values'
assert len(set(smiles)) == len(smiles), 'Duplicate source SMILES; inspect before proceeding'
mols_rdkit = [Chem.MolFromSmiles(s) for s in smiles]
assert all(m is not None for m in mols_rdkit), 'Invalid source SMILES; inspect before proceeding'
pic50 = 6.0 - np.log10(ic50)

# Both MMP endpoint columns match the original molecule SMILES exactly.
# Keep source row identities, including stereoisomers; do not canonicalize or merge.
exact_id = {s: i for i, s in enumerate(smiles)}
endpoints = set(mmp_raw['smiles_1']).union(mmp_raw['smiles_2'])
missing = endpoints.difference(exact_id)
assert not missing, f'MMP endpoints absent from source molecule table: {list(missing)[:5]}'
id1 = mmp_raw['smiles_1'].map(exact_id).to_numpy(int)
id2 = mmp_raw['smiles_2'].map(exact_id).to_numpy(int)
assert np.all(id1 != id2), 'An MMP mapped to the same molecule twice'
delta = pic50[id1] - pic50[id2]
abs_delta = np.abs(delta)
classes = np.where(abs_delta >= CLIFF_CUTOFF, 'AC',
                   np.where(abs_delta <= NONCLIFF_CUTOFF, 'nonAC', 'halfAC'))
class_counts = pd.Series(classes).value_counts().to_dict()
assert class_counts == {'nonAC': 10311, 'halfAC': 1762, 'AC': 521}, class_counts
molecules = pd.DataFrame({'molecule_id': np.arange(len(smiles)), 'smiles': smiles,
                          'source_ic50_uM': ic50, 'pIC50': pic50})
pairs = pd.DataFrame({'source_pair_id': np.arange(len(mmp_raw)), 'molecule_id_1': id1,
                      'molecule_id_2': id2, 'delta_pIC50': delta, 'abs_delta': abs_delta,
                      'derived_class': classes})
for source_column in ('transformation', 'constant_part'):
    if source_column in mmp_raw:
        pairs[source_column] = mmp_raw[source_column].astype(str)
molecules.to_csv(ROOT / 'molecules.csv', index=False)
pairs.to_csv(ROOT / 'source_mmp_pairs.csv', index=False)

fold_of = np.full(len(molecules), -1, dtype=int)
for fold, (_, test_idx) in enumerate(KFold(n_splits=FOLDS, shuffle=True, random_state=SPLIT_SEED).split(molecules)):
    fold_of[test_idx] = fold
assert np.all(fold_of >= 0)
molecules['outer_fold'] = fold_of
molecules.to_csv(ROOT / 'molecule_folds.csv', index=False)
print('Source:', len(molecules), 'molecules;', len(pairs), 'MMPs; classes:', pairs.derived_class.value_counts().to_dict())
print('Molecule fold sizes:', molecules.outer_fold.value_counts().sort_index().to_dict())
dump_json(ROOT / 'raw_sha256.json', {p.name: sha256_file(p) for p in sorted(raw_dir.iterdir())})


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

# Fail before training if any source molecule cannot be featurized.
graphs = []
for i, mol in enumerate(mols_rdkit):
    g = mol_to_whole_graph(mol)
    if g is None:
        raise ValueError(f'Graph featurization failed for molecule_id {i}')
    graphs.append(Data(x=torch.from_numpy(g['x']).float(),
                       edge_index=torch.from_numpy(g['edge_index']).long(),
                       edge_attr=torch.from_numpy(g['edge_attr']).float()))
print('CliffForge graph cache:', len(graphs), 'molecules')

class PairDataset(Dataset):
    def __init__(self, frame):
        self.a = frame.molecule_id_1.to_numpy(int)
        self.b = frame.molecule_id_2.to_numpy(int)
        self.y = frame.delta_pIC50.to_numpy(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return graphs[self.a[idx]], graphs[self.b[idx]], float(self.y[idx])

def pair_collate(batch):
    return (Batch.from_data_list([v[0] for v in batch]),
            Batch.from_data_list([v[1] for v in batch]),
            torch.tensor([v[2] for v in batch], dtype=torch.float32))

def run_relative_seed(fold, seed, train_ids, folder):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    ckpt = folder / 'final.pt'
    if ckpt.exists() and (folder / 'training_log.csv').exists():
        print(f'Fold {fold}, relative seed {seed}: reusing checkpoint')
        return ckpt
    rng_seed = 1000 * fold + seed
    random.seed(rng_seed); np.random.seed(rng_seed); torch.manual_seed(rng_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(rng_seed)
    train_mask = np.zeros(len(molecules), dtype=bool); train_mask[list(train_ids)] = True
    a = pairs.molecule_id_1.to_numpy(int); b = pairs.molecule_id_2.to_numpy(int)
    tr = pairs[train_mask[a] & train_mask[b]]
    assert len(tr), 'Empty relative training MMP set'
    train_loader = DataLoader(PairDataset(tr), batch_size=REL_BATCH_SIZE, shuffle=True,
                              collate_fn=pair_collate, num_workers=0)
    model = SiameseAVES(NODE_DIM, EDGE_DIM, hidden=128, depth=4, dropout=0.1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    loss_fn = nn.SmoothL1Loss()
    history = []
    for ep in range(REL_EPOCHS):
        model.train(); train_sum = train_n = 0
        for ga, gb, y in train_loader:
            ga, gb, y = ga.to(DEVICE), gb.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); pred = model(ga, gb); loss = loss_fn(pred, y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            train_sum += float(loss.detach()) * len(y); train_n += len(y)
        train_loss = train_sum / train_n
        history.append({'epoch': ep, 'train_smooth_l1': train_loss})
        if ep % 5 == 0 or ep == REL_EPOCHS - 1:
            print(f'  fold={fold} relative_seed={seed} epoch={ep} train={train_loss:.4f}')
    torch.save({'state_dict': model.state_dict(), 'fold': fold, 'seed': seed,
                'epochs': REL_EPOCHS, 'n_train_pairs': len(tr),
                'pair_source': 'Dablander supplied MMP table'}, ckpt)
    pd.DataFrame(history).to_csv(folder / 'training_log.csv', index=False)
    dump_json(folder / 'training_meta.json', {'fold': fold, 'seed': seed,
              'train_pairs': len(tr), 'checkpoint': str(ckpt), 'epochs': REL_EPOCHS})
    del model, opt, train_loader
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return ckpt

@torch.no_grad()
def relative_predict(ckpt, a_ids, b_ids, batch_size=256):
    model = SiameseAVES(NODE_DIM, EDGE_DIM, hidden=128, depth=4, dropout=0.1).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False)['state_dict'])
    model.eval(); out = []
    for start in range(0, len(a_ids), batch_size):
        aa = [graphs[int(i)] for i in a_ids[start:start+batch_size]]
        bb = [graphs[int(i)] for i in b_ids[start:start+batch_size]]
        ga = Batch.from_data_list(aa).to(DEVICE)
        gb = Batch.from_data_list(bb).to(DEVICE)
        score = (model(ga, gb) - model(gb, ga)) / 2.0
        out.append(score.cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(out) if out else np.empty(0, dtype=float)


In [ ]:
import shutil
cp_train_cli = str(CP_VENV / 'bin' / 'chemprop_train')
cp_predict_cli = str(CP_VENV / 'bin' / 'chemprop_predict')
assert Path(cp_train_cli).is_file() and Path(cp_predict_cli).is_file(), 'ChemProp 1.x command-line tools were not installed'
print('ChemProp:', cp_versions['chemprop'])
# The previous Mpro baseline used this RDKit fingerprint with chirality disabled.
fingerprints = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) for m in mols_rdkit]
# ChemProp 1.6.1 reloads its own trusted checkpoints with torch.load without the
# weights_only argument; current PyTorch requires this compatibility setting.
cp_env = dict(os.environ, TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD='1')


def knn_test_predictions(train_ids, test_ids, k=5):
    train_ids = np.asarray(sorted(train_ids), dtype=int)
    test_ids = np.asarray(sorted(test_ids), dtype=int)
    train_fps = [fingerprints[int(i)] for i in train_ids]
    train_y = pic50[train_ids]
    result = {}
    for query in test_ids:
        similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(fingerprints[int(query)], train_fps))
        top = np.argsort(-similarities)[:k]
        weights = np.clip(similarities[top], 1e-6, None)
        result[int(query)] = float(np.sum(weights * train_y[top]) / np.sum(weights))
    return result


def run_chemprop_seed(fold, seed, fold_dir, train_csv, test_csv):
    model_dir = fold_dir / 'chemprop' / f'seed_{seed}'
    model_dir.mkdir(parents=True, exist_ok=True)
    best = model_dir / 'fold_0' / 'model_0' / 'model.pt'
    pred_csv = model_dir / 'test_predictions.csv'
    train_log = model_dir / 'train_stdout_stderr.log'
    if not best.exists():
        rng_seed = 1000 * fold + seed
        args = [cp_train_cli, '--data_path', str(train_csv), '--dataset_type', 'regression',
                '--smiles_columns', 'smiles', '--target_columns', 'pIC50',
                '--save_dir', str(model_dir), '--epochs', str(CP_EPOCHS),
                '--num_folds', '1', '--metric', 'rmse', '--extra_metrics', 'r2',
                '--num_workers', '0', '--quiet', '--save_smiles_splits', '--seed', str(rng_seed),
                '--pytorch_seed', str(rng_seed)]
        dump_json(model_dir / 'train_command.json', {'argv': args,
                  'environment_overrides': {'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'}})
        print(f'Fold {fold}, ChemProp seed {seed}: training')
        with open(train_log, 'w') as log:
            result = subprocess.run(args, stdout=log, stderr=subprocess.STDOUT, text=True, env=cp_env)
        if result.returncode:
            tail = '\n'.join(train_log.read_text(errors='replace').splitlines()[-40:])
            raise RuntimeError(f'ChemProp training failed, fold {fold} seed {seed}:\n{tail}')
    else:
        print(f'Fold {fold}, ChemProp seed {seed}: reusing checkpoint')
    assert best.is_file(), f'ChemProp did not save the expected checkpoint: {best}'
    if not pred_csv.exists():
        args = [cp_predict_cli, '--test_path', str(test_csv), '--checkpoint_path', str(best),
                '--preds_path', str(pred_csv), '--smiles_columns', 'smiles', '--num_workers', '0']
        dump_json(model_dir / 'predict_command.json', {'argv': args,
                  'environment_overrides': {'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'}})
        with open(model_dir / 'predict_stdout_stderr.log', 'w') as log:
            result = subprocess.run(args, stdout=log, stderr=subprocess.STDOUT, text=True, env=cp_env)
        if result.returncode:
            tail = '\n'.join((model_dir / 'predict_stdout_stderr.log').read_text(errors='replace').splitlines()[-40:])
            raise RuntimeError(f'ChemProp prediction failed, fold {fold} seed {seed}:\n{tail}')
    predicted = pd.read_csv(pred_csv)
    expected = pd.read_csv(test_csv)
    assert 'pIC50' in predicted and len(predicted) == len(expected), 'Unexpected ChemProp prediction format'
    assert predicted['smiles'].astype(str).tolist() == expected['smiles'].astype(str).tolist(), \
        'ChemProp prediction row order or SMILES changed'
    values = predicted['pIC50'].to_numpy(float)
    assert np.isfinite(values).all(), 'Nonfinite ChemProp predictions'
    return best, values


def make_evaluation_pairs(fold, train_ids, test_ids):
    is_train = np.zeros(len(molecules), dtype=bool); is_train[list(train_ids)] = True
    is_test = np.zeros(len(molecules), dtype=bool); is_test[list(test_ids)] = True
    a = pairs.molecule_id_1.to_numpy(int); b = pairs.molecule_id_2.to_numpy(int)
    strict = is_test[a] & is_test[b]
    known = (is_test[a] & is_train[b]) | (is_train[a] & is_test[b])
    keep = strict | known
    selected = pairs[keep].copy().reset_index(drop=True)
    selected['fold'] = fold
    selected['setting'] = np.where(strict[keep], 'test_test', 'test_train')
    a = selected.molecule_id_1.to_numpy(int)
    b = selected.molecule_id_2.to_numpy(int)
    swap = (selected.setting.to_numpy() == 'test_train') & is_train[a]
    selected['a_id'] = np.where(swap, b, a)  # query in test-train, original first in test-test
    selected['b_id'] = np.where(swap, a, b)  # measured outer-training reference in test-train
    selected['true_delta'] = pic50[selected.a_id.to_numpy(int)] - pic50[selected.b_id.to_numpy(int)]
    selected['true_abs_delta'] = np.abs(selected.true_delta.to_numpy(float))
    selected['class'] = np.where(selected.true_abs_delta >= CLIFF_CUTOFF, 'AC',
                                 np.where(selected.true_abs_delta <= NONCLIFF_CUTOFF, 'nonAC', 'halfAC'))
    assert len(selected) == strict.sum() + known.sum()
    assert (selected['class'].to_numpy() == selected.derived_class.to_numpy()).all()
    return selected


def run_fold(fold):
    folder = ROOT / f'fold_{fold}'; folder.mkdir(exist_ok=True)
    complete = folder / 'fold_complete.json'
    pred_file = folder / 'pair_predictions.csv'
    if complete.exists() and pred_file.exists():
        print(f'Fold {fold}: reusing completed predictions')
        return pd.read_csv(pred_file)
    test_ids = np.where(fold_of == fold)[0]
    outer_train = np.where(fold_of != fold)[0]
    assert len(np.intersect1d(outer_train, test_ids)) == 0
    dump_json(folder / 'split_ids.json', {'fold': fold, 'outer_train': outer_train.tolist(),
              'outer_test': test_ids.tolist()})
    pairs_eval = make_evaluation_pairs(fold, outer_train, test_ids)
    assert (pairs_eval.setting == 'test_test').any() and (pairs_eval.setting == 'test_train').any()
    pairs_eval.to_csv(folder / 'evaluation_pairs_before_predictions.csv', index=False)

    # All models receive the same outer-training pool. ChemProp 1.6.1 creates
    # its original internal random 80/10/10 split; the outer-test CSV has SMILES only.
    train_csv = folder / 'outer_train_molecules.csv'
    pd.DataFrame({'smiles': np.asarray(smiles)[outer_train],
                  'pIC50': pic50[outer_train]}).to_csv(train_csv, index=False)
    test_csv = folder / 'outer_test_smiles_only.csv'
    pd.DataFrame({'smiles': np.asarray(smiles)[test_ids]}).to_csv(test_csv, index=False)

    a_ids = pairs_eval.a_id.to_numpy(int); b_ids = pairs_eval.b_id.to_numpy(int)
    relative = []
    for seed in SEEDS:
        ckpt = run_relative_seed(fold, seed, outer_train,
                                 folder / 'cliffforge' / f'seed_{seed}')
        scores = relative_predict(ckpt, a_ids, b_ids)
        pairs_eval[f'cliffforge_seed_{seed}'] = scores
        relative.append(scores)
    pairs_eval['cliffforge'] = np.mean(relative, axis=0)
    pairs_eval['cliffforge_seed_sd'] = np.std(relative, axis=0)

    cp_each = []
    for seed in SEEDS:
        _, pred = run_chemprop_seed(fold, seed, folder, train_csv, test_csv)
        cp_each.append(pred)
    cp_each = np.asarray(cp_each)
    cp_mean = cp_each.mean(axis=0)
    knn_map = knn_test_predictions(outer_train, test_ids)
    test_pos = {int(mol_id): i for i, mol_id in enumerate(test_ids)}
    test_predictions = pd.DataFrame({'molecule_id': test_ids, 'smiles': np.asarray(smiles)[test_ids],
                                     'measured_pIC50_posthoc': pic50[test_ids],
                                     'chemprop_mean': cp_mean,
                                     'knn_5nn': [knn_map[int(i)] for i in test_ids]})
    for si, seed in enumerate(SEEDS):
        test_predictions[f'chemprop_seed_{seed}'] = cp_each[si]
    test_predictions.to_csv(folder / 'test_molecule_predictions.csv', index=False)

    is_known = pairs_eval.setting.to_numpy() == 'test_train'
    for si, seed in enumerate(SEEDS):
        cp_a = np.asarray([cp_each[si, test_pos[int(i)]] for i in a_ids])
        cp_b = np.asarray([cp_each[si, test_pos[int(i)]] if not is_known[j] else pic50[int(i)]
                           for j, i in enumerate(b_ids)])
        pairs_eval[f'chemprop_seed_{seed}'] = cp_a - cp_b
    pairs_eval['chemprop'] = pairs_eval[[f'chemprop_seed_{s}' for s in SEEDS]].mean(axis=1)
    knn_a = np.asarray([knn_map[int(i)] for i in a_ids])
    knn_b = np.asarray([knn_map[int(i)] if not is_known[j] else pic50[int(i)]
                        for j, i in enumerate(b_ids)])
    pairs_eval['knn_5nn'] = knn_a - knn_b
    assert np.isfinite(pairs_eval[['cliffforge', 'chemprop', 'knn_5nn']].to_numpy(float)).all()
    pairs_eval.to_csv(pred_file, index=False)
    dump_json(complete, {'fold': fold, 'n_outer_train': len(outer_train),
              'n_outer_test': len(test_ids), 'n_test_test_pairs': int((~is_known).sum()),
              'n_test_train_pairs': int(is_known.sum()),
              'finished_utc': datetime.now(timezone.utc).isoformat()})
    print(f'Fold {fold} complete:', int((~is_known).sum()), 'strict pairs,', int(is_known.sum()), 'known-reference pairs')
    return pairs_eval


In [ ]:
MODELS = ('cliffforge', 'chemprop', 'knn_5nn')
BASELINES = ('chemprop', 'knn_5nn')

def safe_ap(frame, model, weights=None):
    if len(frame) == 0 or frame['class'].nunique() < 2:
        return None
    y = (frame['class'].to_numpy() == 'AC').astype(int)
    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        if weights[y == 1].sum() == 0 or weights[y == 0].sum() == 0:
            return None
    return float(average_precision_score(y, np.abs(frame[model].to_numpy(float)), sample_weight=weights))


def detection_report(frame):
    out = {}
    for setting in ('test_test', 'test_train'):
        binary = frame[(frame.setting == setting) & (frame['class'] != 'halfAC')].copy()
        y = (binary['class'].to_numpy() == 'AC').astype(int)
        row = {'n_binary_pairs': len(binary), 'n_cliffs': int(y.sum()),
               'n_noncliffs': int((1-y).sum()),
               'n_halfAC_excluded_only_here': int(((frame.setting == setting) &
                                                   (frame['class'] == 'halfAC')).sum()),
               'prevalence': float(y.mean()) if len(y) else None,
               'models': {}}
        for m in MODELS:
            scores = np.abs(binary[m].to_numpy(float))
            pred = (scores > PRED_CUTOFF).astype(int)
            row['models'][m] = {
                'average_precision': safe_ap(binary, m),
                'roc_auc': float(roc_auc_score(y, scores)) if len(np.unique(y)) == 2 else None,
                'mcc_at_1p5': float(matthews_corrcoef(y, pred)) if len(y) else None,
                'accuracy_at_1p5': float(accuracy_score(y, pred)) if len(y) else None,
                'f1_at_1p5': float(f1_score(y, pred, zero_division=0)) if len(y) else None,
                'precision_at_1p5': float(precision_score(y, pred, zero_division=0)) if len(y) else None,
                'recall_at_1p5': float(recall_score(y, pred, zero_division=0)) if len(y) else None,
                'tp': int(((pred == 1) & (y == 1)).sum()),
                'fp': int(((pred == 1) & (y == 0)).sum()),
                'tn': int(((pred == 0) & (y == 0)).sum()),
                'fn': int(((pred == 0) & (y == 1)).sum()),
            }
        out[setting] = row
    return out


def relative_affinity_report(frame, rank_groups):
    # Qualifying local series have >=3 distinct held-out MMP neighbors.
    eligible = rank_groups[['fold', 'reference_id']].rename(columns={'reference_id': 'b_id'})
    within = frame[frame.setting == 'test_train'].merge(
        eligible, on=['fold', 'b_id'], how='inner', validate='many_to_one')
    settings = {'all_heldout_pairs': frame,
                'test_test': frame[frame.setting == 'test_test'],
                'test_train': frame[frame.setting == 'test_train'],
                'within_series_reference_pairs': within}
    out = {}
    for setting, part in settings.items():
        y = part.true_delta.to_numpy(float)
        row = {'n_pairs': len(part), 'models': {}}
        for m in MODELS:
            pred = part[m].to_numpy(float)
            valid_sign = y != 0
            pearson = float(np.corrcoef(y, pred)[0, 1]) if len(y) > 1 and np.std(y) > 0 and np.std(pred) > 0 else None
            rank = spearmanr(y, pred).statistic if len(y) > 1 and np.unique(y).size > 1 and np.unique(pred).size > 1 else None
            row['models'][m] = {
                'mae_delta_pIC50': float(mean_absolute_error(y, pred)) if len(y) else None,
                'rmse_delta_pIC50': float(np.sqrt(mean_squared_error(y, pred))) if len(y) else None,
                'pearson_r': pearson if pearson is not None and np.isfinite(pearson) else None,
                'spearman_rho': float(rank) if rank is not None and np.isfinite(rank) else None,
                'direction_accuracy_nonzero': float((np.sign(pred[valid_sign]) == np.sign(y[valid_sign])).mean()) if valid_sign.any() else None,
                'n_nonzero_direction_pairs': int(valid_sign.sum()),
            }
        out[setting] = row
    return out


def direction_report(frame):
    out = {}
    for setting in ('test_test', 'test_train'):
        part = frame[(frame.setting == setting) & (frame.true_delta != 0)]
        true_sign = np.sign(part.true_delta.to_numpy(float))
        out[setting] = {'n_nonzero_pairs': len(part), 'accuracy': {
            m: float((np.sign(part[m].to_numpy(float)) == true_sign).mean()) if len(part) else None
            for m in MODELS}}
    return out


def make_rank_groups(known):
    records = []
    for (fold, ref_id), raw in known.groupby(['fold', 'b_id'], sort=True):
        # A source MMP may appear more than once; ranking needs unique candidates.
        g = raw.groupby('a_id', sort=True, as_index=False).agg(
            true_delta=('true_delta', 'first'),
            **{m: (m, 'mean') for m in MODELS})
        if len(g) < 3 or g.true_delta.nunique() < 2:
            continue
        rec = {'fold': int(fold), 'reference_id': int(ref_id), 'n_candidates': len(g)}
        for m in MODELS:
            pred = g[m].to_numpy(float)
            rec[m] = 0.0 if np.unique(pred).size < 2 else float(spearmanr(g.true_delta, pred).statistic)
            assert np.isfinite(rec[m]), f'Undefined Spearman for fold {fold}, ref {ref_id}, {m}'
        records.append(rec)
    result = pd.DataFrame(records)
    if not len(result):
        raise RuntimeError('No evaluable reference neighborhoods for ranking')
    return result


def lead_selection(known):
    selected = []
    metrics = {m: [] for m in MODELS}
    for fold, part in known.groupby('fold', sort=True):
        k = max(1, math.ceil(LEAD_BUDGET * len(part)))
        for m in MODELS:
            ordered = part.sort_values([m, 'source_pair_id'], ascending=[False, True], kind='stable')
            chosen = ordered.head(k).copy()
            chosen['selected_by'] = m
            selected.append(chosen[['fold','source_pair_id','a_id','b_id','true_delta',m,'selected_by']]
                            .rename(columns={m: 'predicted_gain'}))
            metrics[m].append({'fold': int(fold), 'n_candidates': len(part), 'budget': k,
                               'mean_measured_gain': float(chosen.true_delta.mean()),
                               'fraction_actual_gain_gt_0': float((chosen.true_delta > 0).mean()),
                               'fraction_actual_gain_gte_0p5': float((chosen.true_delta >= 0.5).mean())})
    return pd.concat(selected, ignore_index=True), metrics


def bootstrap_contrasts(frame, rank_groups, point, nboot=BOOTSTRAPS):
    strict = frame[(frame.setting == 'test_test') & (frame['class'] != 'halfAC')].reset_index(drop=True)
    known = frame[frame.setting == 'test_train'].reset_index(drop=True)
    rng = np.random.default_rng(SPLIT_SEED + 999)
    n_mol = len(molecules)
    strict_fold = strict.fold.to_numpy(int)
    strict_a = strict.a_id.to_numpy(int); strict_b = strict.b_id.to_numpy(int)
    rank_refs = rank_groups.reference_id.to_numpy(int)
    unique_refs = np.unique(rank_refs)
    known_fold = known.fold.to_numpy(int)
    known_query = known.a_id.to_numpy(int)
    known_y = known.true_delta.to_numpy(float)
    strict_y = (strict['class'].to_numpy() == 'AC').astype(int)
    strict_scores = {m: np.abs(strict[m].to_numpy(float)) for m in MODELS}
    rank_scores = {m: rank_groups[m].to_numpy(float) for m in MODELS}
    lead_order = {(fold, m): np.lexsort((known.source_pair_id.to_numpy(int)[known_fold == fold],
                                        -known[m].to_numpy(float)[known_fold == fold]))
                  for fold in range(FOLDS) for m in MODELS}
    lead_rows = {fold: np.where(known_fold == fold)[0] for fold in range(FOLDS)}
    test_ids_by_fold = {fold: np.where(fold_of == fold)[0] for fold in range(FOLDS)}
    query_ids_by_fold = {fold: np.unique(known_query[known_fold == fold]) for fold in range(FOLDS)}
    keys = [(metric, base) for metric in ('strict_AP', 'within_series_rho', 'lead_top10_gain')
            for base in BASELINES]
    draws = {f'{metric}__vs__{base}': [] for metric, base in keys}

    for iteration in range(nboot):
        # Node bootstrap: every strict MMP receives the product of its two
        # molecule multiplicities. Both endpoints stay together as a pair.
        counts = np.zeros(n_mol, dtype=int)
        for fold in range(FOLDS):
            ids = test_ids_by_fold[fold]
            counts += np.bincount(rng.choice(ids, size=len(ids), replace=True), minlength=n_mol)
        edge_w = counts[strict_a] * counts[strict_b]
        valid_ap = edge_w[strict_y == 1].sum() > 0 and edge_w[strict_y == 0].sum() > 0
        ap = {m: float(average_precision_score(strict_y, strict_scores[m], sample_weight=edge_w))
              if valid_ap else np.nan for m in MODELS}

        # Reference-cluster bootstrap for local analogue ranking. A reference's
        # groups from several outer folds receive the same multiplicity.
        sampled_refs = rng.choice(unique_refs, size=len(unique_refs), replace=True)
        ref_w = np.bincount(sampled_refs, minlength=n_mol)[rank_refs]
        rho = {m: float(np.average(rank_scores[m], weights=ref_w)) for m in MODELS}

        # Query-cluster bootstrap for a fixed top-10%-of-proposals budget.
        query_count = np.zeros(n_mol, dtype=int)
        for fold in range(FOLDS):
            ids = query_ids_by_fold[fold]
            if len(ids):
                query_count += np.bincount(rng.choice(ids, size=len(ids), replace=True), minlength=n_mol)
        lead_w = query_count[known_query]
        lead = {}
        for m in MODELS:
            numerator = denominator = 0.0
            for fold in range(FOLDS):
                idx = lead_rows[fold]
                w = lead_w[idx][lead_order[(fold, m)]]
                y = known_y[idx][lead_order[(fold, m)]]
                budget = max(1, math.ceil(LEAD_BUDGET * int(w.sum())))
                prior = np.r_[0, np.cumsum(w)[:-1]]
                take = np.maximum(0, np.minimum(w, budget - prior))
                numerator += float(np.dot(take, y)); denominator += float(take.sum())
            lead[m] = numerator / denominator if denominator else np.nan

        for metric, base in keys:
            values = {'strict_AP': ap, 'within_series_rho': rho, 'lead_top10_gain': lead}[metric]
            draws[f'{metric}__vs__{base}'].append(values['cliffforge'] - values[base])
        if (iteration + 1) % 500 == 0:
            print('Bootstrap:', iteration + 1, '/', nboot)

    draw_frame = pd.DataFrame(draws)
    draw_frame.insert(0, 'resample', np.arange(nboot))
    draw_frame.to_csv(ROOT / 'bootstrap_paired_differences.csv', index=False)
    alpha = 0.05; n_contrasts = len(keys)
    tail_adjusted = alpha / (2 * n_contrasts)
    result = {}
    for metric, base in keys:
        key = f'{metric}__vs__{base}'
        vals = np.asarray(draws[key], dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) < max(1000, nboot // 2):
            result[key] = {'point_difference': None, 'valid_resamples': len(vals),
                           'ci95': None, 'familywise_adjusted_ci': None,
                           'superiority_supported': False}
            continue
        diff = point[metric]['cliffforge'] - point[metric][base]
        ci95 = np.quantile(vals, [0.025, 0.975]).tolist()
        adjusted = np.quantile(vals, [tail_adjusted, 1-tail_adjusted]).tolist()
        result[key] = {'point_difference': float(diff), 'valid_resamples': len(vals),
                       'ci95': ci95, 'familywise_adjusted_ci': adjusted,
                       'superiority_supported': bool(adjusted[0] > 0)}
    return result


def summarize(all_predictions):
    frame = pd.concat(all_predictions, ignore_index=True)
    assert frame[['cliffforge','chemprop','knn_5nn','true_delta']].notna().all().all()
    frame.to_csv(ROOT / 'all_fold_pair_predictions.csv', index=False)
    known = frame[frame.setting == 'test_train'].copy()
    rank_groups = make_rank_groups(known)
    rank_groups.to_csv(ROOT / 'within_series_reference_groups.csv', index=False)
    selected, lead_by_fold = lead_selection(known)
    selected.to_csv(ROOT / 'lead_top10_selected_pairs.csv', index=False)
    strict = frame[(frame.setting == 'test_test') & (frame['class'] != 'halfAC')]
    lead_point = {m: float(selected[selected.selected_by == m].true_delta.mean()) for m in MODELS}
    lead_secondary_pooled = {}
    for m in MODELS:
        gains = selected.loc[selected.selected_by == m, 'true_delta'].to_numpy(float)
        lead_secondary_pooled[m] = {'n_selected_proposals': len(gains),
                                    'fraction_actual_gain_gt_0': float((gains > 0).mean()),
                                    'fraction_actual_gain_gte_0p5': float((gains >= 0.5).mean())}
    point = {'strict_AP': {m: safe_ap(strict, m) for m in MODELS},
             'within_series_rho': {m: float(rank_groups[m].mean()) for m in MODELS},
             'lead_top10_gain': lead_point}
    detection = detection_report(frame)
    relative_affinity = relative_affinity_report(frame, rank_groups)
    direction = direction_report(frame)
    counts = {'source_molecules': len(molecules), 'source_mmps': len(pairs),
              'strict_test_test_pairs': int((frame.setting == 'test_test').sum()),
              'known_reference_test_train_rows': len(known),
              'strict_binary_cliffs': int((strict['class'] == 'AC').sum()),
              'strict_binary_noncliffs': int((strict['class'] == 'nonAC').sum()),
              'rank_reference_groups': len(rank_groups),
              'rank_unique_reference_molecules': int(rank_groups.reference_id.nunique())}
    print('Primary point metrics:', point)
    print('Evaluation counts:', counts)
    contrasts = bootstrap_contrasts(frame, rank_groups, point)
    summary = {'protocol': protocol, 'counts': counts, 'primary_metric_points': point,
               'primary_paired_contrasts': contrasts,
               'joint_superiority_supported': bool(all(v['superiority_supported'] for v in contrasts.values())),
               'cliff_detection': detection, 'relative_affinity': relative_affinity,
                'potency_direction': direction,
               'lead_secondary_pooled': lead_secondary_pooled,
               'lead_secondary_by_fold': lead_by_fold,
               'fold_counts': [json.loads((ROOT / f'fold_{f}' / 'fold_complete.json').read_text())
                               for f in range(FOLDS)],
               'interpretation': 'Exploratory new five-fold protocol on Dablander Mpro; six planned contrasts; no cross-target generalization.'}
    dump_json(ROOT / 'performance_summary.json', summary)
    lines = ['# CliffForge Dablander five-fold results', '',
             'This bundle contains the exact source CSVs and hashes, molecule folds, outer train/test IDs and ChemProp internal split CSVs,',
             'training logs, all CliffForge and ChemProp checkpoints, per-molecule and per-pair predictions,',
             'paired bootstrap draws, and performance_summary.json.', '',
             '## Primary outcomes', '',
             '| Metric | CliffForge | ChemProp | 5-NN |', '|---|---:|---:|---:|']
    for metric, values in point.items():
        lines.append('| ' + metric + ' | ' + ' | '.join(f'{values[m]:.4f}' if values[m] is not None else 'undefined'
                                                    for m in MODELS) + ' |')
    lines += ['', '## Overall relative affinity (held-out MMPs)', '',
               'Numerical errors use signed ΔpIC50. Test–train pairs use the measured outer-training reference for ChemProp and 5-NN.', '',
               '| Setting | Metric | CliffForge | ChemProp | 5-NN |', '|---|---|---:|---:|---:|']
    for setting, row in relative_affinity.items():
        for metric in ('mae_delta_pIC50', 'rmse_delta_pIC50', 'pearson_r', 'spearman_rho', 'direction_accuracy_nonzero'):
            lines.append('| ' + setting + ' | ' + metric + ' | ' + ' | '.join(
                f"{row['models'][m][metric]:.4f}" if row['models'][m][metric] is not None else 'undefined'
                for m in MODELS) + ' |')
    lines += ['', '## Lead optimization', '',
               'The top 10% of scored candidate/reference proposals are selected separately in each fold.', '',
               '| Metric | CliffForge | ChemProp | 5-NN |', '|---|---:|---:|---:|']
    for metric in ('n_selected_proposals', 'fraction_actual_gain_gt_0', 'fraction_actual_gain_gte_0p5'):
        lines.append('| ' + metric + ' | ' + ' | '.join(
            str(lead_secondary_pooled[m][metric]) if metric == 'n_selected_proposals'
            else f"{lead_secondary_pooled[m][metric]:.4f}" for m in MODELS) + ' |')
    lines += ['', '## Cliff detection', '',
               'Binary metrics exclude halfAC pairs; thresholded metrics use |predicted ΔpIC50| > 1.5.', '',
               '| Setting | Metric | CliffForge | ChemProp | 5-NN |', '|---|---|---:|---:|---:|']
    for setting, row in detection.items():
        for metric in ('average_precision', 'roc_auc', 'accuracy_at_1p5', 'mcc_at_1p5', 'f1_at_1p5', 'precision_at_1p5', 'recall_at_1p5', 'tp', 'fp', 'tn', 'fn'):
            lines.append('| ' + setting + ' | ' + metric + ' | ' + ' | '.join(
                f"{row['models'][m][metric]:.4f}" if isinstance(row['models'][m][metric], float) else str(row['models'][m][metric])
                for m in MODELS) + ' |')
    lines += ['', '## Planned superiority contrasts', '',
              'Each interval is a two-sided Bonferroni-adjusted paired cluster-bootstrap interval',
              'across the six comparisons. Superiority is supported only if the lower bound is positive.', '',
              '| Contrast | Difference | Adjusted interval | Supported |', '|---|---:|---:|---|']
    for name, c in contrasts.items():
        bounds = c['familywise_adjusted_ci']
        lines.append(f"| {name} | {c['point_difference'] if c['point_difference'] is not None else 'undefined'} | "
                     + (f'[{bounds[0]:.4f}, {bounds[1]:.4f}]' if bounds else 'undefined')
                     + f" | {c['superiority_supported']} |")
    lines += ['', f"**All six supported:** {summary['joint_superiority_supported']}", '',
              'Strict test-test and known-reference test-train MMPs are separate tasks. The source paper used',
              'three repeats of two-fold CV; this notebook uses one prespecified five-fold molecule CV.',
              'Pairs can be correlated and a source MMP can appear twice as a known-reference scenario',
              'in opposite query/reference orientations. The archive preserves pair, molecule and fold IDs.', '']
    (ROOT / 'RESULTS_README.md').write_text('\n'.join(lines))
    return summary


## Run, save, and download

The final cell runs all five folds, writes the summary, creates the ZIP, and starts a Colab download. If an exception occurs, it records the traceback and downloads the partial bundle so the logs and completed checkpoints can be inspected. Rerun the cell in the same Colab session to reuse completed folds and checkpoints.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

def create_archive():
    manifest_rows = []
    for p in sorted(ROOT.rglob('*')):
        if p.is_file() and p.name != 'sha256_manifest.csv':
            manifest_rows.append({'relative_path': str(p.relative_to(ROOT)),
                                  'bytes': p.stat().st_size, 'sha256': sha256_file(p)})
    pd.DataFrame(manifest_rows).to_csv(ROOT / 'sha256_manifest.csv', index=False)
    archive = Path('/content/cliffforge_dablander_5fold_results.zip')
    with ZipFile(archive, 'w', ZIP_DEFLATED, compresslevel=6, allowZip64=True) as z:
        for p in sorted(ROOT.rglob('*')):
            if p.is_file():
                z.write(p, arcname=str(Path(ROOT.name) / p.relative_to(ROOT)))
    print(f'Bundle ready: {archive} ({archive.stat().st_size / 1e6:.1f} MB)')
    return archive

failure = None
try:
    fold_predictions = [run_fold(fold) for fold in range(FOLDS)]
    summary = summarize(fold_predictions)
    print('Joint superiority supported:', summary['joint_superiority_supported'])
except Exception as exc:
    failure = exc
    (ROOT / 'RUN_FAILURE_TRACEBACK.txt').write_text(traceback.format_exc())
    print('Run failed; the partial bundle will include logs and checkpoints:', repr(exc))
finally:
    archive = create_archive()
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as download_exc:
        print('Automatic browser download did not start:', download_exc)
        print('The ZIP remains at', archive)
if failure is not None:
    raise failure
